# Stage 1 — Парсинг frame-level YT8M

**Цель:** вытащить покадровые признаки `rgb` (T, 1024) и `audio` (T, 128) из `SequenceExample` — в отличие от старого пайплайна, где на видео был **один** усреднённый вектор.

На выходе — паддированные uint8-массивы длины `L=60` кадров (дробление на float делается в DataLoader), маска реальных длин и метки жанров.

**Формат YT8M frame-level:**
- `context.labels` — int64 список YT8M-меток
- `feature_lists.rgb` — T кадров, каждый 1024 байта (uint8, квантизация min=-2, max=2)
- `feature_lists.audio` — T кадров, каждый 128 байт (аналогично)
- T ∈ [1, 300], 1 кадр ≈ 1 секунда видео

In [1]:
# ============================================================
# 1.0 — Импорты, ClearML и конфиг
# ============================================================
import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
import json, time, warnings
warnings.filterwarnings('ignore')

# ── ClearML: инициализируем В НАЧАЛЕ, чтобы автокапчились пакеты, git,
#    stdout/stderr и все matplotlib-графики ноутбука.
from clearml import Task
task = Task.init(
    project_name="Video Classifier",
    task_name="01_parse_frame_level",
    task_type=Task.TaskTypes.data_processing,
    reuse_last_task_id=False,
)

# ── Пути ─────────────────────────────────────────────────────
BASE_DIR   = Path('../data')
DATA_DIR   = BASE_DIR / 'video' / 'frame_validate'   # frame-level shards
OUT_DIR    = BASE_DIR / 'frame_processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Параметры парсинга — пробрасываем через task.connect,
#    чтобы в ClearML UI их можно было переопределить при клонировании задачи.
parse_cfg = {
    'L'            : 60,      # фиксированная длина последовательности (кадров ≈ секунд)
    'FRAME_STRIDE' : 1,       # шаг между кадрами (1 — подряд, 2 — через один)
    'MAX_PER_CLASS_CAP': 5000,
    'BALANCE_RATIO_OF_MIN': 3,  # cap = min(5000, min_count * 3)
    'SEED'         : 42,
    'DIM_RGB'      : 1024,
    'DIM_AUD'      : 128,
}
task.connect(parse_cfg, name='parse_config')

L            = parse_cfg['L']
FRAME_STRIDE = parse_cfg['FRAME_STRIDE']
DIM_RGB      = parse_cfg['DIM_RGB']
DIM_AUD      = parse_cfg['DIM_AUD']

# ── Маппинг YT8M label_id → наш жанр (из старого пайплайна) ──
LABEL_MAP = {
    0: 'Gaming',   1: 'Gaming',   27: 'Gaming',  35: 'Gaming',
    36:'Gaming',   42: 'Gaming',  81: 'Gaming',
    3: 'Music',    4:  'Music',   9:  'Music',   10: 'Music',
    13:'Music',    14: 'Music',   28: 'Music',   31: 'Music',
    33:'Music',    34: 'Music',   37: 'Music',   38: 'Music',
    40:'Music',    41: 'Music',   86: 'Music',
    5: 'Animation', 16:'Animation',
    2: 'Vehicles', 7:  'Vehicles',17: 'Vehicles',19: 'Vehicles',
    30:'Vehicles', 44: 'Vehicles',45: 'Vehicles',
    12:'Sports',   43: 'Sports',  82: 'Sports',
    15:'Animals',  18: 'Animals',
    11:'Food',     20: 'Food',    22: 'Food',
    29:'Food',     32: 'Food',
    8: 'Dance',
    6: 'Performance',
    21:'Tech',     23: 'Tech',    24: 'Tech',
    39:'Beauty',
    25:'Film',
}
GENRES    = sorted(set(LABEL_MAP.values()))
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}

# Приоритет при мультилейбле — редкие выше
GENRE_PRIORITY = {
    'Beauty':0,'Film':1,'Dance':2,'Animals':3,'Sports':4,'Tech':5,
    'Performance':6,'Food':7,'Animation':8,'Vehicles':9,'Music':10,'Gaming':11,
}

print(f'DATA_DIR    : {DATA_DIR}')
print(f'OUT_DIR     : {OUT_DIR}')
print(f'L (frames)  : {L}')
print(f'Жанров      : {len(GENRES)} — {GENRES}')
print(f'ClearML task: {task.id}')

ClearML Task: created new task id=6e2d77668fbf481fb1beaad0af42e378
ClearML results page: https://app.clear.ml/projects/f6756b373a194d77b25caf2f40ed8950/experiments/6e2d77668fbf481fb1beaad0af42e378/output/log
2026-05-17 12:52:02,992 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
DATA_DIR    : ../data/video/frame_validate
OUT_DIR     : ../data/frame_processed
L (frames)  : 60
Жанров      : 12 — ['Animals', 'Animation', 'Beauty', 'Dance', 'Film', 'Food', 'Gaming', 'Music', 'Performance', 'Sports', 'Tech', 'Vehicles']
ClearML task: 6e2d77668fbf481fb1beaad0af42e378


In [2]:
# ============================================================
# 1.1 — Проверка входных файлов
# ============================================================
tfrecord_files = sorted(DATA_DIR.glob('validate*.tfrecord'))
print(f'Найдено TFRecord файлов: {len(tfrecord_files)}')
if not tfrecord_files:
    raise FileNotFoundError(
        f'Нет frame-level файлов в {DATA_DIR}.\n'
        f'Скачай командой:\n'
        f'  cd {DATA_DIR} && partition=2/frame/validate mirror=eu shard=1,200 \\\n'
        f'      uv run python ../../download.py'
    )

# Быстрая проверка первого файла — убедимся что frame-level
ds = tf.data.TFRecordDataset(str(tfrecord_files[0]))
for raw in ds.take(1):
    seq = tf.train.SequenceExample()
    seq.ParseFromString(raw.numpy())
    n_frames = len(seq.feature_lists.feature_list.get('rgb', tf.train.FeatureList()).feature)
    assert n_frames > 0, 'Feature_lists пусты — это video-level, не frame-level!'
    print(f'OK: первое видео имеет {n_frames} кадров, feature_lists = {list(seq.feature_lists.feature_list.keys())}')

Найдено TFRecord файлов: 101
OK: первое видео имеет 149 кадров, feature_lists = ['rgb', 'audio']


I0000 00:00:1779011569.345010  777376 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


In [3]:
# ============================================================
# 1.2 — Парсинг: собираем padded uint8 тензоры
# ============================================================
# Храним данные как uint8 — это нативная квантизация YT8M,
# экономит 4× памяти. Dequantize делается на лету в DataLoader:
#   x = byte * (4/255) - (2 - 2/255/2)  ≈  byte*0.01569 - 1.99216
#
# Паддинг: L=60 кадров. Если T<L → справа нули, маска отметит
# реальную длину. Если T>L → берём первые L кадров (с шагом).

rgb_list    = []   # каждый элемент (L, 1024) uint8
audio_list  = []   # каждый элемент (L, 128) uint8
length_list = []   # реальная длина после паддинга/обрезания
label_list  = []   # индекс жанра
id_list     = []   # YT8M anonymized id
skipped     = 0

t0 = time.time()
for file_idx, tf_file in enumerate(tfrecord_files):
    ds = tf.data.TFRecordDataset(str(tf_file))
    for raw in ds:
        seq = tf.train.SequenceExample()
        seq.ParseFromString(raw.numpy())

        # --- метки ---
        lbls = list(seq.context.feature['labels'].int64_list.value)
        matched = [LABEL_MAP[l] for l in lbls if l in LABEL_MAP]
        if not matched:
            skipped += 1
            continue
        genre   = min(matched, key=lambda g: GENRE_PRIORITY[g])
        y_idx   = GENRE2IDX[genre]

        # --- кадры rgb ---
        rgb_frames = seq.feature_lists.feature_list['rgb'].feature
        aud_frames = seq.feature_lists.feature_list['audio'].feature
        T = min(len(rgb_frames), len(aud_frames))
        if T == 0:
            skipped += 1
            continue

        # берём первые L кадров со stride
        idxs    = list(range(0, T, FRAME_STRIDE))[:L]
        real_len = len(idxs)

        rgb_arr = np.zeros((L, DIM_RGB), dtype=np.uint8)
        aud_arr = np.zeros((L, DIM_AUD), dtype=np.uint8)
        for i, j in enumerate(idxs):
            rgb_arr[i] = np.frombuffer(rgb_frames[j].bytes_list.value[0], dtype=np.uint8)
            aud_arr[i] = np.frombuffer(aud_frames[j].bytes_list.value[0], dtype=np.uint8)

        rgb_list.append(rgb_arr)
        audio_list.append(aud_arr)
        length_list.append(real_len)
        label_list.append(y_idx)
        id_list.append(seq.context.feature['id'].bytes_list.value[0].decode('utf-8', errors='ignore'))

    if (file_idx + 1) % 5 == 0 or (file_idx + 1) == len(tfrecord_files):
        dt = time.time() - t0
        print(f'   [{file_idx+1:>3}/{len(tfrecord_files)}] '
              f'собрано {len(label_list):>6}  пропущено {skipped:>5}  '
              f'({dt:.1f}s)')

# сборка в массивы
X_rgb   = np.stack(rgb_list,   axis=0)   # (N, L, 1024) uint8
X_audio = np.stack(audio_list, axis=0)   # (N, L, 128)  uint8
lengths = np.array(length_list, dtype=np.int32)
y       = np.array(label_list,  dtype=np.int64)
ids     = np.array(id_list)

print(f'\nИтого:')
print(f'  X_rgb   : {X_rgb.shape}  dtype={X_rgb.dtype}  ~{X_rgb.nbytes/1e6:.1f} MB')
print(f'  X_audio : {X_audio.shape}  dtype={X_audio.dtype}  ~{X_audio.nbytes/1e6:.1f} MB')
print(f'  lengths : {lengths.shape}  min={lengths.min()} max={lengths.max()} mean={lengths.mean():.1f}')
print(f'  y       : {y.shape}  классов={len(np.unique(y))}')
print(f'  пропущено (нет нужных меток / пустое видео): {skipped}')

   [  5/101] собрано   1131  пропущено   376  (0.3s)
   [ 10/101] собрано   2243  пропущено   741  (0.6s)
   [ 15/101] собрано   3301  пропущено  1072  (1.0s)
   [ 20/101] собрано   4439  пропущено  1422  (1.3s)
   [ 25/101] собрано   5518  пропущено  1801  (1.6s)
   [ 30/101] собрано   6619  пропущено  2141  (1.9s)
   [ 35/101] собрано   7694  пропущено  2512  (2.2s)
   [ 40/101] собрано   8800  пропущено  2884  (2.5s)
   [ 45/101] собрано   9850  пропущено  3245  (2.8s)
   [ 50/101] собрано  10997  пропущено  3589  (3.1s)
   [ 55/101] собрано  12112  пропущено  3937  (3.4s)
   [ 60/101] собрано  13173  пропущено  4289  (3.7s)
   [ 65/101] собрано  14268  пропущено  4638  (4.0s)
   [ 70/101] собрано  15391  пропущено  4978  (4.3s)
   [ 75/101] собрано  16488  пропущено  5345  (4.6s)
   [ 80/101] собрано  17584  пропущено  5701  (4.9s)
   [ 85/101] собрано  18688  пропущено  6056  (5.2s)
   [ 90/101] собрано  19770  пропущено  6389  (5.5s)
   [ 95/101] собрано  20917  пропущено  6706  

In [4]:
# ============================================================
# 1.3 — Статистика классов + отсечение малых
# ============================================================
counts = np.bincount(y, minlength=len(GENRES))
N = len(y)
print(f'Распределение классов (N={N:,}):')
print(f'  {"жанр":<14} {"N":>7}   {"%":>6}')
print('  ' + '─'*34)
for i, g in enumerate(GENRES):
    pct = counts[i] / N * 100
    bar = '█' * int(pct / 2)
    print(f'  {g:<14} {counts[i]:>7,}   {pct:>5.1f}%  {bar}')
print(f'  imbalance ratio: {counts.max()/max(counts.min(),1):.2f}×')

Распределение классов (N=22,218):
  жанр                 N        %
  ──────────────────────────────────
  Animals            943     4.2%  ██
  Animation        1,933     8.7%  ████
  Beauty             319     1.4%  
  Dance            1,417     6.4%  ███
  Film               436     2.0%  
  Food             1,017     4.6%  ██
  Gaming           4,953    22.3%  ███████████
  Music            4,729    21.3%  ██████████
  Performance        988     4.4%  ██
  Sports           1,262     5.7%  ██
  Tech               633     2.8%  █
  Vehicles         3,588    16.1%  ████████
  imbalance ratio: 15.53×


In [5]:
# ============================================================
# 1.4 — Балансировка: cap per class
# ============================================================
MIN_COUNT     = max(counts.min(), 1)
MAX_PER_CLASS = min(parse_cfg['MAX_PER_CLASS_CAP'], MIN_COUNT * parse_cfg['BALANCE_RATIO_OF_MIN'])
print(f'min class={MIN_COUNT:,}  cap={MAX_PER_CLASS:,}')

rng = np.random.default_rng(parse_cfg['SEED'])
keep = []
for c in range(len(GENRES)):
    idx = np.where(y == c)[0]
    if len(idx) > MAX_PER_CLASS:
        idx = rng.choice(idx, MAX_PER_CLASS, replace=False)
    keep.append(idx)
keep = np.concatenate(keep)
rng.shuffle(keep)

X_rgb   = X_rgb[keep]
X_audio = X_audio[keep]
lengths = lengths[keep]
y       = y[keep]
ids     = ids[keep]

counts_bal = np.bincount(y, minlength=len(GENRES))
print(f'\nПосле балансировки: N={len(y):,}, imbalance={counts_bal.max()/max(counts_bal.min(),1):.2f}×')
print(f'  {"жанр":<14} {"до":>7} {"после":>7}')
for i, g in enumerate(GENRES):
    print(f'  {g:<14} {counts[i]:>7,} {counts_bal[i]:>7,}')

# ── ClearML: распределение классов как bar chart (report_histogram → столбцы)
logger = task.get_logger()
logger.report_histogram(
    title="Class Distribution",
    series="Raw (before balancing)",
    values=list(counts),
    xlabels=GENRES,
    iteration=0,
    yaxis="Video count",
)
logger.report_histogram(
    title="Class Distribution",
    series="Balanced (after cap)",
    values=list(counts_bal),
    xlabels=GENRES,
    iteration=0,
    yaxis="Video count",
)
logger.report_single_value("imbalance_ratio_raw", float(counts.max()/max(counts.min(),1)))
logger.report_single_value("imbalance_ratio_bal", float(counts_bal.max()/max(counts_bal.min(),1)))
logger.report_single_value("n_samples_final", int(len(y)))

min class=319  cap=957

После балансировки: N=9,987, imbalance=3.00×
  жанр                до   после
  Animals            943     943
  Animation        1,933     957
  Beauty             319     319
  Dance            1,417     957
  Film               436     436
  Food             1,017     957
  Gaming           4,953     957
  Music            4,729     957
  Performance        988     957
  Sports           1,262     957
  Tech               633     633
  Vehicles         3,588     957


In [6]:
# ============================================================
# 1.5 — Сохранение
# ============================================================
np.save(OUT_DIR / 'X_rgb.npy',   X_rgb)     # (N, L, 1024) uint8
np.save(OUT_DIR / 'X_audio.npy', X_audio)   # (N, L, 128)  uint8
np.save(OUT_DIR / 'lengths.npy', lengths)
np.save(OUT_DIR / 'y.npy',       y)
np.save(OUT_DIR / 'ids.npy',     ids)

pd.DataFrame([{'label_idx':i,'category':g} for i,g in IDX2GENRE.items()]) \
  .to_csv(OUT_DIR / 'label_map.csv', index=False)

config = {
    'n_samples'    : int(len(y)),
    'n_classes'    : len(GENRES),
    'L'            : L,
    'frame_stride' : FRAME_STRIDE,
    'dim_rgb'      : DIM_RGB,
    'dim_audio'    : DIM_AUD,
    'genres'       : GENRES,
    'genre2idx'    : GENRE2IDX,
    'max_per_class': int(MAX_PER_CLASS),
    # dequantize: x = (byte + 0.5) * (max-min)/255 + min,   min=-2, max=2
    'dequant_scale': 4.0 / 255.0,
    'dequant_bias' : 4.0/(2*255) - 2.0,
    # ── важные счётчики классов (нужны downstream для sanity-check)
    'class_counts_raw': {GENRES[i]: int(counts[i])     for i in range(len(GENRES))},
    'class_counts_bal': {GENRES[i]: int(counts_bal[i]) for i in range(len(GENRES))},
}
with open(OUT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print('=' * 55)
print('STAGE 1 — COMPLETE')
print('=' * 55)
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name:<20} {p.stat().st_size/1024**2:>8.2f} MB')

STAGE 1 — COMPLETE
  X_audio.npy             73.15 MB
  X_rgb.npy              585.18 MB
  config.json              0.00 MB
  ids.npy                  0.15 MB
  label_map.csv            0.00 MB
  lengths.npy              0.04 MB
  y.npy                    0.08 MB


In [7]:
# ============================================================
# 1.6 — Закрытие ClearML Task (появится в разделе Tasks как completed)
# ============================================================
task.close()

In [7]:
# ============================================================
# 1.6 — Выгрузка в ClearML Dataset
# ============================================================
# Используем `clearml.Dataset` (а не просто upload_artifact) — это даёт:
#   • версионирование датасета (каждый ре-ран = новая версия с автоинкрементом)
#   • дедупликацию файлов между версиями (пере-заливается только diff)
#   • явный lineage: downstream stage3 укажет этот датасет как parent
#   • одну строку для любого коллеги: Dataset.get(...).get_local_copy()
#
# Raw TFRecord'ы (4 ГБ) НЕ льём — квоту SaaS пожалеем, они публичные и
# тривиально скачиваются через data/download.py.

from clearml import Dataset

processed_ds = Dataset.create(
    dataset_name="yt8m-frame-processed",
    dataset_project="Video Classifier",
    dataset_tags=[f"L={L}", f"stride={FRAME_STRIDE}", f"n={len(y)}"],
    description=(
        f"Parsed YT8M frame-level features, padded to L={L}, "
        f"balanced (cap {MAX_PER_CLASS} per class). "
        f"{len(y)} samples × {len(GENRES)} classes."
    ),
    use_current_task=True,   # прикрепляет датасет к текущей задаче 01_parse_frame_level
)
processed_ds.add_files(str(OUT_DIR))
processed_ds.upload(show_progress=True)
processed_ds.finalize()     # запечатывает версию (immutable)

print(f"✓ Dataset uploaded: id={processed_ds.id}  name={processed_ds.name}")
print(f"  use downstream:  Dataset.get(dataset_id='{processed_ds.id}').get_local_copy()")

ClearML results page: https://app.clear.ml/projects/d1663e2e99e141ec94e7305572cd1fdb/experiments/b5a119ed5ea74e348ad6a8b75c6c7cb7/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/d1663e2e99e141ec94e7305572cd1fdb/experiments/b5a119ed5ea74e348ad6a8b75c6c7cb7
Generating SHA2 hash for 7 files


100%|██████████| 7/7 [00:00<00:00,  8.63it/s]


Hash generation completed
Uploading dataset changes (1 files compressed to 561.83 MiB) to https://files.clear.ml


▊                                3% | 15.00/561.83 MB [00:00<00:30, 18.12MB/s]: 

Uploading dataset changes (6 files compressed to 67.32 MiB) to https://files.clear.ml


█                                4% | 20.00/561.83 MB [00:01<01:02,  8.73MB/s]: 
█▌                               5% | 30.00/561.83 MB [00:02<00:48, 11.00MB/s]: 
█▊                               6% | 35.00/561.83 MB [00:02<00:44, 11.82MB/s]: 
██████▉                          22% | 15.00/67.32 MB [00:01<00:05,  8.95MB/s]: 
██▏                              7% | 40.00/561.83 MB [00:04<01:10,  7.43MB/s]: 
██▍                              8% | 45.00/561.83 MB [00:04<01:06,  7.76MB/s]: 
██▋                              9% | 50.00/561.83 MB [00:05<01:08,  7.48MB/s]: 
██▉                             10% | 55.00/561.83 MB [00:06<01:17,  6.54MB/s]: 
███▍                            12% | 65.00/561.83 MB [00:08<01:36,  5.15MB/s]: 
███▋                            12% | 70.00/561.83 MB [00:09<01:28,  5.53MB/s]: 
████                            13% | 75.00/561.83 MB [00:10<01:21,  5.95MB/s]: 
████▎                           14% | 80.00/561.83 MB [00:11<01:17,  6.22MB/s]: 
███████████████████████████▋

File compression and upload completed: total size 629.15 MiB, 2 chunk(s) stored (average size 314.58 MiB)
✓ Dataset uploaded: id=b5a119ed5ea74e348ad6a8b75c6c7cb7  name=01_parse_frame_level
  use downstream:  Dataset.get(dataset_id='b5a119ed5ea74e348ad6a8b75c6c7cb7').get_local_copy()
